# Protogen EDA
This Dataset is a collection of proteomics collected from all patients during BL (Baseline/0 Months) and 6M (6 Months after starting therapy)
not done yet


# Setup & Imports

In [ ]:
print("Loading Libraries")
# Core Libraries
import pandas as pd # excel tools
import numpy as np # math
import matplotlib.pyplot as plt # visualizing data
import os # file/folder operations
import difflib # fuzzy matching for write errors
print("Done loading")

# Load Data from Excel Sheet
Loading from the Sheet and organising it through their sheets

In [ ]:
#reading from excel file
all_sheets = pd.read_excel('../datasets/Protogen_RA_MAP_16_05_21.xlsx', sheet_name=None)
#extracting sheet names
print(all_sheets.keys())
df_Samples = all_sheets['LIS_PG665-P01 RA MAP Samples Ex']
df_Samples_Annotation = all_sheets['Sample annotation']
#columns and rows per sheet
for name, sheet in all_sheets.items():
    print(f'{name}: {sheet.shape[0]} rows x {sheet.shape[1]} columns')

## First look at Data
Since we dont have a glossary for this we have to rely on the .info() function
Here we take a look at the structure of the Data (cell/column names) by taking a random example

In [ ]:
#display(df_Samples_Annotation.info())
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(df_Samples_Annotation.sample().T)   


Since df_Samples_Annotation is just a copy of the clinical data we can ignore it and move on to the protein analytics


In [ ]:
df_Samples.info()
#has 163x597 = 97 311 values in total

In [ ]:

display(df_Samples.sample().T)

# Info
A quick look tells us df_Samples is structured like this
Protein-,GeneID with Symbol and Name
Then all the Samples from the patients and their assessment month (PatientID_ASSESSEMENT)

Thus we have the value of a speicific protein for all of the patients, this is helpful since we can link it with other datasets that reveal more information of a patient

# Data Science Stuff
In this section we calculate descriptive statistics for each protein across all patients, both at baseline and after 6 months of therapy. From there we generate several visualizations:

- **Top 20 highest reactivity at baseline** — which proteins are most strongly expressed at the start
- **Top 20 highest variance at baseline** — which proteins show the biggest patient-to-patient differences
- **Top 20 biggest decrease/increase BL → M6** — which proteins respond most to treatment
- **Median vs variance scatter** — identifies proteins that are both highly reactive and highly variable, making them interesting candidates for further analysis

All plots and a summary report are saved to the `../reports/` directory.

> The first 3 cells (column setup, validation helper, stats preparation) need to be run before the plotting and analysis cells work

In [ ]:
# create copy to not affect main source
df_Samples_copy = df_Samples.copy()

# seperate by timestamp BL and M6
bl_cols = [col for col in df_Samples.columns if col.endswith('_BL')]
m6_cols = [col for col in df_Samples.columns if col.endswith('_M6')]

print(f'Baseline samples: {len(bl_cols)}')
print(f'6-month samples: {len(m6_cols)}')

In [ ]:
# helper function for fuzzy matching later on has to be ran once
def validate_protein(protein, df=df_Samples):
    # check if protein exists, suggest close matches if not
    proteins = df['Gene Symbol'].dropna().unique()
    if protein in proteins:
        return True

    # the difflib method checks all 'protein' entries 'proteins' and returns max 5 suggestions that have a 60% (0.6) similarity
    suggestions = difflib.get_close_matches(protein, proteins, n=5, cutoff=0.6)
    if suggestions:
        print(f'Protein "{protein}" not found. Did you mean: {", ".join(suggestions)}?')
    else:
        print(f'Protein "{protein}" not found and no close matches')
    return False

print("method in cell will run when called upon e.g. in topresponder cell")

In [ ]:
# preparing dataframes
# Stats needed for BL
stats_bl = pd.DataFrame({
    'Gene Symbol': df_Samples['Gene Symbol'],
    'Gene Name': df_Samples['Gene Name'],
    'mean_BL': df_Samples[bl_cols].mean(axis=1),
    'median_BL': df_Samples[bl_cols].median(axis=1),
    'max_BL': df_Samples[bl_cols].max(axis=1),
    'std_BL': df_Samples[bl_cols].std(axis=1)
})

# Stats needed for M6
stats_m6 = pd.DataFrame({
    'Gene Symbol': df_Samples['Gene Symbol'],
    'Gene Name': df_Samples['Gene Name'],
    'mean_M6': df_Samples[m6_cols].mean(axis=1),
    'median_M6': df_Samples[m6_cols].median(axis=1),
    'max_M6': df_Samples[m6_cols].max(axis=1),
    'std_M6': df_Samples[m6_cols].std(axis=1)
})

# Differences in the timeframe
stats_combined = pd.DataFrame({
    'Gene Symbol': df_Samples['Gene Symbol'],
    'mean_BL': stats_bl['mean_BL'],
    'mean_M6': stats_m6['mean_M6'],
    'delta': stats_m6['mean_M6'] - stats_bl['mean_BL'],
    'delta_pct': ((stats_m6['mean_M6'] - stats_bl['mean_BL']) / stats_bl['mean_BL'] * 100).round(1)
})
# pair patients that have both BL and M6 timestamps
bl_patients = {col.replace('_BL', '') for col in df_Samples.columns if col.endswith('_BL')}
m6_patients = {col.replace('_M6', '') for col in df_Samples.columns if col.endswith('_M6')}
paired_patients = sorted(bl_patients & m6_patients)
print(f'Patients with paired data: {len(paired_patients)}')

print("dataframes prepped")

In [ ]:
# plot visualization
# First plot
# Top 20 Proteins reacting during Baseline
# creates a reports directory or overwrites it if it exists
os.makedirs('../reports', exist_ok=True)
fig, axes = plt.subplots(2, 2, figsize=(16, 14)) # formatting

top_mean = stats_bl.sort_values('mean_BL', ascending=False).head(20) # show dataframe with the 20 descending values
#formatting
axes[0, 0].barh(top_mean['Gene Symbol'][::-1], top_mean['mean_BL'][::-1], color='steelblue')
axes[0, 0].set_xlabel('Mean reactivity')
axes[0, 0].set_title('Top 20 highest reactivity at baseline')

# Second plot
# Variance of the same proteins
top_std = stats_bl.sort_values('std_BL', ascending=False).head(20)
#formatting
axes[0, 1].barh(top_std['Gene Symbol'][::-1], top_std['std_BL'][::-1], color='coral')
axes[0, 1].set_xlabel('Standard deviation')
axes[0, 1].set_title('Top 20 highest variance at baseline')

# Third plot
# shows the biggest decrease of protein reacctivity from BL -> M6
top_decrease = stats_combined.sort_values('delta').head(20)
#formatting
colors = ['forestgreen' if d < 0 else 'firebrick' for d in top_decrease['delta'][::-1]]
axes[1, 0].barh(top_decrease['Gene Symbol'][::-1], top_decrease['delta'][::-1], color=colors)
axes[1, 0].set_xlabel('Change (BL → M6)')
axes[1, 0].set_title('Top 20 biggest decrease after therapy')
axes[1, 0].axvline(x=0, color='black', linewidth=0.5)

# fourth plot
# shows the biggest increase of protein reacctivity from BL -> M6
top_increase = stats_combined.sort_values('delta', ascending=False).head(20)
#formatting
colors = ['firebrick' if d > 0 else 'forestgreen' for d in top_increase['delta'][::-1]]
axes[1, 1].barh(top_increase['Gene Symbol'][::-1], top_increase['delta'][::-1], color=colors)
axes[1, 1].set_xlabel('Change (BL → M6)')
axes[1, 1].set_title('Top 20 biggest increase after therapy')
axes[1, 1].axvline(x=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig('../reports/protogen_overview.png', dpi=150, bbox_inches='tight')
plt.show()

# fifth plot part 1
# median vs variance as a scatter for BL
plt.figure(figsize=(12, 8))
plt.scatter(stats_bl['median_BL'], stats_bl['std_BL'], alpha=0.5, color='steelblue')

# save interesting protein stats
interesting = stats_bl[
    (stats_bl['std_BL'] > stats_bl['std_BL'].quantile(0.9)) |
    (stats_bl['median_BL'] > stats_bl['median_BL'].quantile(0.9))
]
for _, row in interesting.iterrows():
    plt.annotate(row['Gene Symbol'], (row['median_BL'], row['std_BL']),
                fontsize=8, alpha=0.7)

plt.xlabel('Median reactivity at baseline')
plt.ylabel('Standard deviation at baseline')
plt.title('Median vs variance — top right = high reactivity + high variability')
plt.tight_layout()
plt.savefig('../reports/protogen_median_vs_variance_BL.png', dpi=150, bbox_inches='tight')
plt.show()

# fifth plot part 2 
# median vs variance as a scatter for M6
# uncomment the """ if you wish to ignore it
#"""
plt.figure(figsize=(12, 8))
plt.scatter(stats_m6['median_M6'], stats_m6['std_M6'], alpha=0.5, color='steelblue')

interesting = stats_m6[
    (stats_m6['std_M6'] > stats_m6['std_M6'].quantile(0.9)) |
    (stats_m6['median_M6'] > stats_m6['median_M6'].quantile(0.9))
]

for _, row in interesting.iterrows():
    plt.annotate(row['Gene Symbol'], (row['median_M6'], row['std_M6']),
                fontsize=8, alpha=0.7)

plt.xlabel('Median reactivity at 6 months')
plt.ylabel('Standard deviation at 6 months')
plt.title('Median vs variance — top right = high reactivity + high variability')
plt.tight_layout()
plt.savefig('../reports/protogen_median_vs_variance_M6.png', dpi=150, bbox_inches='tight')
plt.show()
#"""

with open('../reports/showcase.txt', 'w') as f:
    f.write('=== Protogen Protein Analysis ===\n\n')
    f.write(f'Total proteins: {len(stats_bl)}\n')
    f.write(f'Baseline samples: {len(bl_cols)}\n')
    f.write(f'6-month samples: {len(m6_cols)}\n\n')
    f.write('--- Top 20 highest reactivity at baseline ---\n')
    f.write(top_mean[['Gene Symbol', 'mean_BL', 'median_BL', 'std_BL']].to_string(index=False))
    f.write('\n\n--- Top 20 highest variance at baseline ---\n')
    f.write(top_std[['Gene Symbol', 'mean_BL', 'median_BL', 'std_BL']].to_string(index=False))
    f.write('\n\n--- Top 20 biggest decrease BL to M6 ---\n')
    f.write(top_decrease[['Gene Symbol', 'mean_BL', 'mean_M6', 'delta', 'delta_pct']].to_string(index=False))
    f.write('\n\n--- Top 20 biggest increase BL to M6 ---\n')
    f.write(top_increase[['Gene Symbol', 'mean_BL', 'mean_M6', 'delta', 'delta_pct']].to_string(index=False))

print('\nSaved plots and report to ../reports/')

In [ ]:
# function to show the distribution of a specific protein from baseline to 6 months
def plot_protein_distribution(protein, df=df_Samples):
    # fuzzy matching from helper function
    if not validate_protein(protein, df):
        return
    
    row = df[df['Gene Symbol'] == protein]
    gene_name = row['Gene Name'].values[0]
    
    # extract values for both timepoints
    bl_values = row[bl_cols].values.flatten()
    m6_values = row[m6_cols].values.flatten()
    
    # plot side by side
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # formatting for baseline
    axes[0].bar(range(len(bl_values)), sorted(bl_values, reverse=True), color='steelblue')
    axes[0].set_xlabel('Patients (sorted)')
    axes[0].set_ylabel('Reactivity')
    axes[0].set_title(f'{protein} ({gene_name}) — Baseline')
    # formatting for month 6
    axes[1].bar(range(len(m6_values)), sorted(m6_values, reverse=True), color='coral')
    axes[1].set_xlabel('Patients (sorted)')
    axes[1].set_ylabel('Reactivity')
    axes[1].set_title(f'{protein} ({gene_name}) — 6 months')
    
    plt.tight_layout()
    plt.show()

# example calls
plot_protein_distribution('NONO')
plot_protein_distribution('VIM')
plot_protein_distribution('MCM2')

In [ ]:
def plot_top_responders(protein, df=df_Samples, n=10):
    if not validate_protein(protein, df):
        return
    
    row = df[df['Gene Symbol'] == protein]
    gene_name = row['Gene Name'].values[0]
    
    # build a small dataframe with BL, M6 and delta per patient
    data = []
    for pid in paired_patients:
        bl_val = row[f'{pid}_BL'].values[0]
        m6_val = row[f'{pid}_M6'].values[0]
        data.append({'patient': pid, 'BL': bl_val, 'M6': m6_val, 'delta': bl_val - m6_val})
    
    paired_df = pd.DataFrame(data)
    
    # top n decrease and increase
    groups = [
        ('decrease', paired_df.sort_values('delta', ascending=False).head(n), 'steelblue'),
        ('increase', paired_df.sort_values('delta', ascending=True).head(n), 'coral')
    ]
    
    # helper to plot a slope chart
    def slope_plot(subset, color, label):
        fig, ax = plt.subplots(figsize=(10, 7))
        for _, r in subset.iterrows():
            ax.plot([0, 1], [r['BL'], r['M6']], marker='o', color=color, alpha=0.7)
            ax.annotate(r['patient'], (1.02, r['M6']), fontsize=8, va='center')
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Baseline', '6 Months'])
        ax.set_ylabel('Reactivity')
        ax.set_title(f'{protein} ({gene_name}) — Top {n} responders (biggest {label})')
        ax.set_xlim(-0.2, 1.3)
        plt.tight_layout()
        plt.show()
    
    # plot and print stats for both groups
    for label, subset, color in groups:
        slope_plot(subset, color, label)
        print(f'\n=== {protein} ({gene_name}) — Top {n} {label} ===')
        print(f'Mean BL:    {subset["BL"].mean():.2f}')
        print(f'Mean M6:    {subset["M6"].mean():.2f}')
        print(f'Mean delta: {subset["delta"].mean():.2f}')

# example calls
plot_top_responders('NONO', n=10)